# Contextual AI: MCP Tools that Encode Platform Engineering Knowledge

Build three **Elastic Agent Builder** tools that encode platform engineering knowledge into reusable functions.
Once created, any developer with an MCP-compatible editor (Claude Code, Cursor, VS Code) can ask
*"Is it safe to merge this PR?"* and get a reasoned, evidence-based answer drawn from live production signals.

Pipeline: Create sample environment (deploy annotations + SLO) > Build three tools (`get_endpoint_health`, `get_recent_deploys`, `get_slo_status`) > Test each tool > Connect via MCP.

## Setup

### Prerequisite: generate APM traffic

This notebook needs a live APM service to query. If your cluster does not already have
APM data ingested, use `elastic/apm-integration-testing` to generate synthetic traffic
from an opbeans demo app pointed at your existing cluster:

```bash
git clone https://github.com/elastic/apm-integration-testing.git
cd apm-integration-testing

APM_SERVER_URL=https://YOUR_APM_SERVER_URL \
APM_TOKEN=YOUR_APM_SECRET_TOKEN \
python3 ./scripts/compose.py start 9.3.0 \
  --no-kibana \
  --no-elasticsearch \
  --with-opbeans-node
```

Let it run for 5-10 minutes to populate `traces-apm-*`. The opbeans-node container
reports as service `opbeans-node` in APM (used as `SERVICE_NAME` below).

### Python dependencies

In [14]:
%pip install requests dotenv -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

KIBANA_URL = os.getenv("KIBANA_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
SERVICE_NAME = "opbeans-node"

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

print(f"Kibana URL: {KIBANA_URL}")
print(f"Service: {SERVICE_NAME}")

## Create Deploy Annotations

Deploy annotations mark when deployments happened so the agent can correlate latency changes with code releases.
Each annotation is indexed into the APM `observability-annotations` index.

We create two annotations to simulate a small deploy history.

In [ ]:
annotations = [
    {
        "@timestamp": "2026-04-24T09:00:00.000Z",
        "service": {"version": "2.0.9", "environment": "production"},
        "message": "Deployed v2.0.9 - fixed cart calculation bug",
    },
    {
        "@timestamp": "2026-04-24T11:00:00.000Z",
        "service": {"version": "2.1.0", "environment": "production"},
        "message": "Deployed v2.1.0 - updated product aggregation logic",
    },
]

for ann in annotations:
    response = requests.post(
        f"{KIBANA_URL}/api/apm/services/{SERVICE_NAME}/annotation",
        headers=headers,
        json=ann,
    )
    print(f"Annotation '{ann['service']['version']}': {response.status_code}")
    print(response.json())

## Create the SLO

Create an APM Availability SLO for the checkout endpoint with a 30-day rolling window
and 99.5% availability target.

Save the `id` from the response. You will need it when testing `get_slo_status`.

In [17]:
slo_payload = {
    "name": "Checkout Availability",
    "description": "Checkout endpoint availability SLO",
    "indicator": {
        "type": "sli.apm.transactionErrorRate",
        "params": {
            "service": SERVICE_NAME,
            "environment": "production",
            "transactionType": "request",
            "transactionName": "POST /checkout",
            "index": "metrics-apm.transaction.1m-*",
        },
    },
    "timeWindow": {"duration": "30d", "type": "rolling"},
    "objective": {"target": 0.995},
    "budgetingMethod": "occurrences",
}

response = requests.post(
    f"{KIBANA_URL}/api/observability/slos",
    headers=headers,
    json=slo_payload,
)
print(f"SLO created: {response.status_code}")
slo_data = response.json()
print(json.dumps(slo_data, indent=2))

SLO_ID = slo_data.get("id")
print(f"\nSLO_ID = {SLO_ID}")

SLO created: 200
{
  "id": "b03b6845-20be-495b-9eb5-c0eb0da8c332"
}

SLO_ID = b03b6845-20be-495b-9eb5-c0eb0da8c332


## Tool 1: get_endpoint_health

Returns the current health of a service endpoint: error rate, latency percentiles (p50/p95/p99), and throughput.

**The key insight:** the `description` field carries the interpretation rules
(thresholds, baselines, deploy correlation). This is the runbook encoded in the tool.

In [ ]:
endpoint_health_tool = {
    "id": "get_endpoint_health",
    "type": "esql",
    "description": (
        "Returns the current health of a service endpoint: error rate, latency percentiles "
        "(p50/p95/p99), and throughput. Use this tool to assess whether a service is healthy "
        "before making changes. Interpretation guide: error rate below 0.5% is healthy, "
        "0.5-1% is elevated (check for recent deploys), above 1% indicates a problem. "
        "For latency, compare p99 against the service baseline: checkout is typically "
        "under 500ms, product-search under 200ms. A sudden p99 spike within 15 minutes "
        "of a deploy suggests the deploy caused a regression. Pass timeWindow as an ES|QL "
        "duration string such as '1 hour' or '6 hours', and windowMinutes as the same window "
        "expressed in minutes (used to compute throughput per minute)."
    ),
    "tags": ["apm", "reliability", "health"],
    "configuration": {
        "query": (
            "FROM traces-apm-* "
            "| WHERE service.name == ?serviceName "
            "AND @timestamp >= NOW() - ?timeWindow "
            "AND transaction.duration.us IS NOT NULL "
            "| STATS total_transactions = COUNT(*), "
            'error_count = SUM(CASE(event.outcome == "failure", 1, 0)), '
            "p50_latency_ms = PERCENTILE(transaction.duration.us, 50) / 1000, "
            "p95_latency_ms = PERCENTILE(transaction.duration.us, 95) / 1000, "
            "p99_latency_ms = PERCENTILE(transaction.duration.us, 99) / 1000 "
            "BY service.name "
            "| EVAL error_rate_pct = ROUND(error_count / total_transactions * 100, 2) "
            "| EVAL throughput_per_min = ROUND(total_transactions / ?windowMinutes, 1)"
        ),
        "params": {
            "serviceName": {
                "type": "keyword",
                "description": "The APM service name to check (e.g., ecommerce-app)",
            },
            "timeWindow": {
                "type": "keyword",
                "description": "Time window to analyze, in ES|QL duration format (e.g., 30 minutes, 1 hour, 6 hours)",
            },
            "windowMinutes": {
                "type": "integer",
                "description": "Time window in minutes, used to calculate throughput per minute",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=endpoint_health_tool,
)
print(f"Tool created: {response.status_code}")
print(response.json())

Tool created: 200
{'id': 'get_endpoint_health', 'type': 'esql', 'description': "Returns the current health of a service endpoint: error rate, latency percentiles (p50/p95/p99), and throughput. Use this tool to assess whether a service is healthy before making changes. Interpretation guide: error rate below 0.5% is healthy, 0.5-1% is elevated (check for recent deploys), above 1% indicates a problem. For latency, compare p99 against the service baseline: checkout is typically under 500ms, product-search under 200ms. A sudden p99 spike within 15 minutes of a deploy suggests the deploy caused a regression. Pass timeWindow as an ES|QL duration string such as '1 hour' or '6 hours', and windowMinutes as the same window expressed in minutes (used to compute throughput per minute).", 'tags': ['apm', 'reliability', 'health'], 'configuration': {'query': 'FROM traces-apm-* | WHERE service.name == ?serviceName AND @timestamp >= NOW() - ?timeWindow AND transaction.duration.us IS NOT NULL | STATS tot

### Test `get_endpoint_health`

Execute the tool against live APM data. Output values depend on your traffic patterns.

In [ ]:
response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools/_execute",
    headers=headers,
    json={
        "tool_id": "get_endpoint_health",
        "tool_params": {"serviceName": SERVICE_NAME},
    },
)
print(json.dumps(response.json(), indent=2))

## Tool 2: get_recent_deploys

Returns the recent deployment history for a service. Deploy context is critical
because most production issues correlate with code changes.

> **Index note:** APM deploy annotations are stored in the `observability-annotations` index
> (no leading dot). The default can be overridden with `xpack.observability.annotations.index`
> in `kibana.yml`. Before running, verify the index exists in your cluster:
> `GET _cat/indices/observability-annotations?v`

In [ ]:
recent_deploys_tool = {
    "id": "get_recent_deploys",
    "type": "esql",
    "description": (
        "Returns the deployment history for a service over the LAST 24 HOURS, including "
        "version numbers, timestamps, and deploy messages. Use this tool to understand the "
        "deployment timeline when assessing service health. Key patterns: if a deploy happened "
        "within the last 15 minutes, elevated error rates or latency may be expected (warm-up "
        "period). If metrics degraded immediately after a deploy, the deploy is the likely "
        "cause. Multiple deploys in a short window (under 2 hours) increase risk because it "
        "becomes harder to isolate which change caused an issue."
    ),
    "tags": ["apm", "deploys", "change-tracking"],
    "configuration": {
        "query": (
            "FROM observability-annotations "
            "| WHERE service.name == ?serviceName "
            "AND @timestamp >= NOW() - 24 hours "
            "| SORT @timestamp DESC "
            "| KEEP @timestamp, service.version, service.environment, message "
            "| LIMIT 10"
        ),
        "params": {
            "serviceName": {
                "type": "keyword",
                "description": "The APM service name to check deploy history for",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=recent_deploys_tool,
)
print(f"Tool created: {response.status_code}")
print(response.json())

In [ ]:
response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools/_execute",
    headers=headers,
    json={
        "tool_id": "get_recent_deploys",
        "tool_params": {"serviceName": SERVICE_NAME},
    },
)
print(json.dumps(response.json(), indent=2))

## Tool 3: get_slo_status

Returns the current SLO **burn rate** over the last hour: how fast the service is
consuming error budget relative to the allowed threshold.

Burn rate is the SRE-grade metric for the "is it safe to act now?" question.
A burn rate of 1.0 means you are consuming budget at exactly the allowed pace.
Fast-burn alerts typically trigger above 14x.

> **Index note:** SLO SLI data lives in `.slo-observability.sli-v*` (with leading dot,
> no trailing date suffix). The major version depends on your Stack release
> (e.g., `v3.6` in 9.3). Run `GET _cat/indices/.slo-observability.*?v`
> to confirm the pattern in your cluster.

In [ ]:
slo_status_tool = {
    "id": "get_slo_status",
    "type": "esql",
    "description": (
        "Returns the current SLO burn rate for a service over the last hour. The response "
        "includes: SLI value (current performance), error budget target, and burn rate "
        "percentage. The burn rate tells you how fast the service is consuming error budget "
        "relative to the allowed threshold. Interpretation: a burn rate below 100% means the "
        "service is consuming budget slower than the limit, which is sustainable. Between "
        "100-200%, the service is burning budget faster than planned, so proceed with caution. "
        "Above 200%, the service is burning budget at double the allowed rate, so delay "
        "non-critical changes. Above 500%, investigate immediately. Note: this measures the "
        "*current* burn rate over the last hour, not cumulative budget consumption over the "
        "full SLO window. A temporarily high burn rate does not mean the overall budget is exhausted."
    ),
    "tags": ["slo", "reliability", "budget"],
    "configuration": {
        "query": (
            "FROM .slo-observability.sli-v* "
            "| WHERE slo.id == ?sloId "
            "AND @timestamp >= NOW() - 1 hour "
            "| STATS sli_value = AVG(slo.numerator) / AVG(slo.denominator) "
            "BY slo.id, slo.name "
            "| EVAL error_budget_target = 0.995 "
            "| EVAL burn_rate_pct = ROUND((1 - sli_value) / (1 - error_budget_target) * 100, 1)"
        ),
        "params": {
            "sloId": {
                "type": "keyword",
                "description": "The SLO identifier. Use the SLO ID for the service you are evaluating.",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=slo_status_tool,
)
print(f"Tool created: {response.status_code}")
print(response.json())

In [23]:
response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools/_execute",
    headers=headers,
    json={
        "tool_id": "get_slo_status",
        "tool_params": {"sloId": SLO_ID},
    },
)
print(json.dumps(response.json(), indent=2))

{
  "results": [
    {
      "tool_result_id": "kR7DkB",
      "type": "error",
      "data": {
        "message": "verification_exception\n\tRoot causes:\n\t\tverification_exception: Found 1 problem\nline 1:1: Unknown index [.slo-observability.sli-v*-*]"
      }
    }
  ]
}


## Connect to your editor via MCP

With all three tools created, they are automatically exposed through the MCP server endpoint
at `/api/agent_builder/mcp`. Configure your editor's MCP client to connect:

### Claude Code / Cursor (`~/.cursor/mcp.json` or Claude Code settings)

```json
{
  "mcpServers": {
    "elastic-agent-builder": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "https://YOUR_KIBANA_URL/api/agent_builder/mcp",
        "--header",
        "Authorization: ApiKey YOUR_BASE64_API_KEY"
      ]
    }
  }
}
```

Restart the editor. Ask *"What Elastic tools do you have?"*. The agent should list
`get_endpoint_health`, `get_recent_deploys`, and `get_slo_status`.

## Cleanup

In [12]:
for tool_id in ["get_endpoint_health", "get_recent_deploys", "get_slo_status"]:
    response = requests.delete(
        f"{KIBANA_URL}/api/agent_builder/tools/{tool_id}",
        headers=headers,
    )
    print(f"Deleted {tool_id}: {response.status_code}")

Deleted get_endpoint_health: 200
Deleted get_recent_deploys: 404
Deleted get_slo_status: 404


In [13]:
if SLO_ID:
    response = requests.delete(
        f"{KIBANA_URL}/api/observability/slos/{SLO_ID}",
        headers=headers,
    )
    print(f"Deleted SLO {SLO_ID}: {response.status_code}")

Deleted SLO dd067a20-cca4-4fab-91da-31c2d5406d25: 204
